# AKANE E2-7：地理でもう一度 ＋ 世界で使われている間違い探しと比べる

noteに書く前の確かめ。
1. **地理でもう一度**：「パリはフランスの市？」のテスト（前にやったもの）で、Gemmaにルールを作らせ、ルールも疑う網で比べる。
   ルールの質問の例："Is it true that no place in France is in Spain?"（フランスにあって、しかもスペインにある場所はない、は本当？）
2. **他の方法と比べる**：同じ○×問題を、別の言い方でも聞く（言い換え）。答えがぶれるかどうかも、間違い探しの手がかりとして比べる。

**このノートでやること**：Gemmaに約1万6千問（短い質問）を聞いて、答えをドライブに保存するだけ。元の○×問題の答えは前に保存したものを使う。計算は手元のPCで行う。
**時間の目安**（T4 GPU）：10〜20分。計画は sims/e2_selfrule/PLAN_E2-7.md（結果を見る前に固定）。

## 0. 準備
1. 「ランタイム」→「ランタイムのタイプを変更」→ **T4 GPU**。
2. 鍵マークの `HF_TOKEN`（前と同じもの）にアクセスを許可。
3. 「ランタイム」→「すべてのセルを実行」。

In [ ]:
# ===== 設定 =====
USE_MOCK_MOUTH = False      # True: 作り物の口（動作確認用。結果は研究に使わない）
SAVE_TO_DRIVE  = True
MODEL_ID = "google/gemma-2-2b-it"
DATA_SEED = 0

In [ ]:
import os, json, time, itertools, collections
import numpy as np
for pkg, mod in [("nltk", "nltk"), ("wordfreq", "wordfreq"), ("geonamescache", "geonamescache")]:
    try: __import__(mod)
    except ImportError: os.system(f"pip -q install {pkg}")
import geonamescache
OUT = "/content/E2_results" if os.path.exists("/content") else "E2_results"
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/AKANE_アーキテクチャ/E2_results"
os.makedirs(OUT, exist_ok=True)
print("保存先:", OUT)

## 1. ものの分類のテスト（前と同じもの）

In [ ]:
import nltk
from nltk.corpus import wordnet as wn
from wordfreq import zipf_frequency
try: nltk.data.find("corpora/wordnet")
except LookupError:
    try: nltk.data.find("corpora/wordnet.zip")
    except LookupError: nltk.download("wordnet", quiet=True)
# 大分類（7つ）と種類（25）。種類は WordNet の見出し、英語の言い方、大分類
CONT = {"animal": "animal", "plant": "plant", "food": "food", "artifact": "man-made object",
        "activity": "activity", "condition": "medical condition", "feeling": "feeling"}
CATS = {"mammal.n.01": ("mammal", "animal"), "bird.n.01": ("bird", "animal"), "fish.n.01": ("fish", "animal"),
        "insect.n.01": ("insect", "animal"), "reptile.n.01": ("reptile", "animal"),
        "tree.n.01": ("tree", "plant"), "flower.n.01": ("flower", "plant"),
        "vegetable.n.01": ("vegetable", "food"), "edible_fruit.n.01": ("fruit", "food"), "beverage.n.01": ("drink", "food"),
        "dish.n.02": ("dish", "food"), "baked_goods.n.01": ("baked good", "food"),
        "tool.n.01": ("tool", "artifact"), "vehicle.n.01": ("vehicle", "artifact"),
        "musical_instrument.n.01": ("musical instrument", "artifact"), "weapon.n.01": ("weapon", "artifact"),
        "furniture.n.01": ("furniture", "artifact"), "garment.n.01": ("clothing", "artifact"),
        "container.n.01": ("container", "artifact"), "building.n.01": ("building", "artifact"),
        "sport.n.01": ("sport", "activity"), "dance.n.01": ("dance", "activity"), "game.n.01": ("game", "activity"),
        "disease.n.01": ("disease", "condition"), "emotion.n.01": ("emotion", "feeling")}
# どちらとも言える大分類の組（偽の問題に使わない）：植物と食べ物、動物と食べ物
AMBIG = {frozenset(("plant", "food")), frozenset(("animal", "food"))}
MASS = {"furniture", "clothing"}
art = lambda w: w if w in MASS else ("an " if w[0] in "aeiou" else "a ") + w
lemma = lambda s: s.lemmas()[0].name()
by_co = collections.defaultdict(list)          # 種類 → ことば
for sid in CATS:
    S = wn.synset(sid)
    for leaf in S.closure(lambda s: s.hyponyms()):
        w = lemma(leaf)
        if "_" in w or not w.isalpha() or not w.islower() or zipf_frequency(w, "en") < 2.5: continue
        if len(wn.synsets(w)) != 1: continue        # 意味が1つしかない語だけ（取り違えを避ける）
        anc = set(leaf.closure(lambda s: s.hypernyms()))
        hits = [c for c in CATS if wn.synset(c) in anc]
        if hits == [sid]: by_co[sid].append(w)
rng = np.random.default_rng(DATA_SEED)
COUNTRIES = [k for k in CATS if len(set(by_co[k])) >= 4]
STMT, GOLD, KIND, CITY = [], [], [], []
idx = {}
def add(key, text, gold, kind, city=None):
    if key in idx: return idx[key]
    idx[key] = len(STMT); STMT.append(text); GOLD.append(gold); KIND.append(kind); CITY.append(city)
    return idx[key]
cont_of = {k: CATS[k][1] for k in COUNTRIES}
same_cont = collections.defaultdict(list)
for k in COUNTRIES: same_cont[cont_of[k]].append(k)
others = lambda z: [w for w in CONT if w != z and frozenset((w, z)) not in AMBIG]
# 種類→大分類
for k in COUNTRIES:
    z = cont_of[k]; fz = rng.choice(others(z))
    for zz in (z, fz):
        add(("CK", k, zz), f"{art(CATS[k][0]).capitalize()} is a kind of {CONT[zz]}.", int(zz == z), "種類→大分類")
# ことば→種類、ことば→大分類（1つの種類から多くても30語）
CITIES = []
for k in COUNTRIES:
    ws = sorted(set(by_co[k]))
    pick = list(rng.choice(ws, min(30, len(ws)), replace=False))
    for name in pick:
        z = cont_of[k]; CITIES.append(name)
        same = [w for w in same_cont[z] if w != k]     # 偽の種類は、なるべく同じ大分類から2つ
        pool = same if len(same) >= 2 else same + [w for w in COUNTRIES if cont_of[w] in others(z)]
        fks = list(rng.choice(pool, 2, replace=False))
        for kk in [k] + fks:
            add(("CC", name, kk), f"{name.capitalize()} is a kind of {CATS[kk][0]}.", int(kk == k), "ことば→種類", name)
        for zz in (z, rng.choice(others(z))):
            add(("CL", name, zz), f"{name.capitalize()} is a kind of {CONT[zz]}.", int(zz == z), "ことば→大分類", name)
GOLD = np.array(GOLD)
DUMMY = len(STMT)   # 「両立しない」ルールの結論に使う、いつも0の素子
# ルール：(前提1, 前提2, 結論)。結論がDUMMYなら「前提が両方1ではいけない」
RULES = set()
for (t, a, b), i in list(idx.items()):
    if t == "CC":
        for zz in CONT:
            if ("CK", b, zz) in idx and ("CL", a, zz) in idx:
                RULES.add((i, idx[("CK", b, zz)], idx[("CL", a, zz)]))     # ことばが種類に入り、種類が大分類に入る → ことばはその大分類
def excl(keys):
    ids = [idx[k] for k in keys if k in idx]
    for p, q in itertools.combinations(sorted(ids), 2): RULES.add((p, q, DUMMY))
for name in set(CITIES):
    excl([("CC", name, k) for k in COUNTRIES]); excl([("CL", name, z) for z in CONT])
for k in COUNTRIES:
    excl([("CK", k, z) for z in CONT])
RULES = sorted(RULES)
# ルールが正解で全部守られているか（守られていなければデータの誤り）
G = np.append(GOLD, 0)
bad = sum(1 for p, q, h in RULES if G[p] == 1 and G[q] == 1 and G[h] == 0)
print(f"種類 {len(COUNTRIES)}、ことば {len(set(CITIES))}、問題 {len(STMT)}（本物 {GOLD.sum()}）、ルール {len(RULES)}本、正解が破るルール {bad}")
print("例:", " | ".join(STMT[i] for i in [0, 1, 60, 61, 62, 63, 64]))
WN_STMT = list(STMT)

## 2. 地理のテスト（前と同じもの）

In [ ]:
gc = geonamescache.GeonamesCache()
CONT = {"AF": "Africa", "AS": "Asia", "EU": "Europe", "NA": "North America", "OC": "Oceania", "SA": "South America"}
SKIP = {"RU", "TR", "EG", "KZ", "AZ", "GE", "AM", "CY", "ID", "PG", "TL"}
cos = {k: v for k, v in gc.get_countries().items() if v["continentcode"] in CONT and k not in SKIP}
allc = list(gc.get_cities().values())
name_cnt = collections.Counter(c["name"] for c in allc)
cnames = {v["name"] for v in cos.values()}
by_co = collections.defaultdict(list)
for c in allc:
    if c["countrycode"] in cos and name_cnt[c["name"]] == 1 and c["name"] not in cnames and c["name"].isascii():
        by_co[c["countrycode"]].append(c)
COUNTRIES = [k for k in sorted(by_co) if len(by_co[k]) >= 4]
rng = np.random.default_rng(DATA_SEED)
STMT, GOLD, KIND, CITY = [], [], [], []
idx = {}
def add(key, text, gold, kind, city=None):
    if key in idx: return idx[key]
    idx[key] = len(STMT); STMT.append(text); GOLD.append(gold); KIND.append(kind); CITY.append(city)
    return idx[key]
cont_of = {k: cos[k]["continentcode"] for k in COUNTRIES}
same_cont = collections.defaultdict(list)
for k in COUNTRIES: same_cont[cont_of[k]].append(k)
others = lambda z: [w for w in CONT if w != z]
# 国→大陸
for k in COUNTRIES:
    z = cont_of[k]; fz = rng.choice(others(z))
    for zz in (z, fz):
        add(("CK", k, zz), f"{cos[k]['name']} is a country in {CONT[zz]}.", int(zz == z), "国→大陸")
# 市→国、市→大陸
CITIES = []
for k in COUNTRIES:
    cs = sorted(by_co[k], key=lambda c: -c["population"])
    pick = [cs[0]] + list(rng.choice(cs[1:], 3, replace=False))
    for c in pick:
        name = c["name"]; z = cont_of[k]; CITIES.append(name)
        fks = [w for w in same_cont[z] if w != k]
        fks = list(rng.choice(fks, min(2, len(fks)), replace=False))
        for kk in [k] + fks:
            add(("CC", name, kk), f"{name} is a city in {cos[kk]['name']}.", int(kk == k), "市→国", name)
        for zz in (z, rng.choice(others(z))):
            add(("CL", name, zz), f"{name} is located in {CONT[zz]}.", int(zz == z), "市→大陸", name)
GOLD = np.array(GOLD)
DUMMY = len(STMT)   # 「両立しない」ルールの結論に使う、いつも0の素子
# ルール：(前提1, 前提2, 結論)。結論がDUMMYなら「前提が両方1ではいけない」
RULES = set()
for (t, a, b), i in list(idx.items()):
    if t == "CC":
        for zz in CONT:
            if ("CK", b, zz) in idx and ("CL", a, zz) in idx:
                RULES.add((i, idx[("CK", b, zz)], idx[("CL", a, zz)]))     # 市が国にあり、国が大陸にある → 市はその大陸
def excl(keys):
    ids = [idx[k] for k in keys if k in idx]
    for p, q in itertools.combinations(sorted(ids), 2): RULES.add((p, q, DUMMY))
for name in set(CITIES):
    excl([("CC", name, k) for k in COUNTRIES]); excl([("CL", name, z) for z in CONT])
for k in COUNTRIES:
    excl([("CK", k, z) for z in CONT])
RULES = sorted(RULES)
# ルールが正解で全部守られているか（守られていなければデータの誤り）
G = np.append(GOLD, 0)
bad = sum(1 for p, q, h in RULES if G[p] == 1 and G[q] == 1 and G[h] == 0)
print(f"国 {len(COUNTRIES)}、市 {len(set(CITIES))}、問題 {len(STMT)}（本物 {GOLD.sum()}）、ルール {len(RULES)}本、正解が破るルール {bad}")
print("例:", STMT[0], "|", STMT[2], "|", STMT[5])
GEO_STMT = list(STMT)

## 3. 質問を作る
- 地理のルール：同じ市の問題に一緒に出てくる国・大陸の組。"Is it true that no place in X is in Y?"（順番を入れ替えて2回）
- 言い換えA："Is the following statement false?"（はい＝間違い）
- 言い換えB："… Is this true or false?"（True/False で答える）

In [ ]:
PAIRS = set()
for name in set(CITIES):
    L = [("K", k) for k in COUNTRIES if ("CC", name, k) in idx] + [("Z", z) for z in CONT if ("CL", name, z) in idx]
    for p, q in itertools.combinations(L, 2): PAIRS.add(tuple(sorted((p, q))))
for k in COUNTRIES:
    for p, q in itertools.combinations([("Z", z) for z in CONT if ("CK", k, z) in idx], 2): PAIRS.add(tuple(sorted((p, q))))
PAIRS = sorted(PAIRS)
gname = lambda lab: cos[lab[1]]["name"] if lab[0] == "K" else CONT[lab[1]]
def truth_excl(p, q):
    if p[0] == q[0]: return True
    k, z = (p, q) if p[0] == "K" else (q, p)
    return cont_of[k[1]] != z[1]
QS = []   # (種類, 番号, 順番, 答え方, 質問)  答え方 YN=Yes/No, TF=True/False
for j, (p, q) in enumerate(PAIRS):
    QS.append(("georule", j, 0, "YN", f"Is it true that no place in {gname(p)} is in {gname(q)}? Answer Yes or No only."))
    QS.append(("georule", j, 1, "YN", f"Is it true that no place in {gname(q)} is in {gname(p)}? Answer Yes or No only."))
for dom, SS in [("wn", WN_STMT), ("geo", GEO_STMT)]:
    for i, s in enumerate(SS):
        QS.append((dom + "_paraA", i, 0, "YN", f"Is the following statement false? Answer Yes or No only.\nStatement: {s}"))
        QS.append((dom + "_paraB", i, 0, "TF", f"{s} Is this true or false? Answer True or False only."))
print(f"地理のルールの組 {len(PAIRS)}（両立する {sum(not truth_excl(p, q) for p, q in PAIRS)}）、質問 {len(QS)}")
for k in ["georule", "wn_paraA", "wn_paraB"]: print(next(q[-1] for q in QS if q[0] == k))

## 4. Gemmaに聞く（保存するので2回目からは飛ばせる）

In [ ]:
RQ = [q[-1] for q in QS]
rcache = f"{OUT}/E2-7_answers_{'MOCK' if USE_MOCK_MOUTH else MODEL_ID.split('/')[-1]}.json"
if os.path.exists(rcache):
    d = json.load(open(rcache))
    if d["questions"] != RQ: raise SystemExit("保存済みの答えと質問が合わない。ファイルを消してやり直す")
    RLOGIT = np.array(d["logit"]); print("保存済みの答えを読み込んだ:", rcache)
elif USE_MOCK_MOUTH:
    RLOGIT = np.random.default_rng(7).normal(0, 3, len(QS))
else:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from google.colab import userdata
    tok = AutoTokenizer.from_pretrained(MODEL_ID, token=userdata.get("HF_TOKEN"), padding_side="left")
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, token=userdata.get("HF_TOKEN"),
                                                 torch_dtype=torch.float16, device_map="cuda").eval()
    T = {w: tok.encode(w, add_special_tokens=False)[0] for w in ["Yes", "No", "True", "False"]}
    def rprompt(s):
        return tok.apply_chat_template([{"role": "user", "content": s}], tokenize=False, add_generation_prompt=True)
    RLOGIT = np.zeros(len(QS)); B = 64; t = time.time()
    for i in range(0, len(QS), B):
        chunk = QS[i:i + B]
        enc = tok([rprompt(q[-1]) for q in chunk], return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        with torch.no_grad():
            last = model(**enc).logits[:, -1, :].float()
        yn = (last[:, T["Yes"]] - last[:, T["No"]]).cpu().numpy(); tf = (last[:, T["True"]] - last[:, T["False"]]).cpu().numpy()
        RLOGIT[i:i + B] = np.where([q[3] == "TF" for q in chunk], tf, yn)
        if (i // B) % 25 == 0: print(f"{i + B}/{len(QS)}  {time.time() - t:.0f}秒", flush=True)
if not os.path.exists(rcache):
    json.dump({"model": "MOCK" if USE_MOCK_MOUTH else MODEL_ID, "qs": QS, "questions": RQ, "logit": RLOGIT.tolist(),
               "geo_pairs": PAIRS, "geo_truth": [truth_excl(p, q) for p, q in PAIRS]}, open(rcache, "w"), ensure_ascii=False)
    print("保存:", rcache)

## 5. ざっと確かめる（ここまでで終わり。結果のファイルを手元で読みます）

In [ ]:
v = np.zeros((len(PAIRS), 2))
for (k, j, o, a, q), l in zip(QS, RLOGIT):
    if k == "georule": v[j, o] = l
g = v.mean(1) > 0; tt = np.array([truth_excl(p, q) for p, q in PAIRS])
print(f"地理のルール：正しい判断 {(g == tt).mean():.1%}、見逃し {(tt & ~g).sum()}/{tt.sum()}、間違ったルール {(~tt & g).sum()}/{(~tt).sum()}")
for dom, SS, GG in [("wn", WN_STMT, None), ("geo", GEO_STMT, GOLD)]:
    b = np.array([l for (k, *_), l in zip(QS, RLOGIT) if k == dom + "_paraB"])
    if GG is not None: print(f"地理の言い換えB（True/False）だけの正解率 {((b > 0) == GG).mean():.3f}")